# 09 · Sessions and State: Memory, and What Survives a Crash

> **Kernel:** `Phase 5 · OpenAI Agents SDK` · **Needs:** Ollama.

Agents are stateless: each run sees only what it's given. **Sessions** give a
conversation memory: the SDK loads the history before each run and saves new
items after each turn, keyed by a session id. This notebook covers sessions in
depth, then answers a question every production system must answer: **when the
process dies mid-run, what's kept and what's lost?** We test it with real,
killed processes.

**You will learn**
- `SQLiteSession`: conversation memory across runs and restarts
- Session operations: read, undo (`pop_item`), clear, and limit how much history is sent
- Sessions with handoffs: memory is shared, but *you* choose the starting agent
- **What survives a crash**: sessions save after every turn, and what that means for resuming
- The three kinds of state in the SDK, and when you need your own checkpoints

**Prerequisites:** notebooks 02, 06 and 08.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 02_OPENAI_AGENTS_SDK folder
sys.path.insert(0, str(SECTION))

import subprocess, sys, textwrap, pathlib
from agents import Agent, Runner, SQLiteSession, SessionSettings
from common import get_model

model = get_model()
WORK = pathlib.Path('checkpoints')
WORK.mkdir(exist_ok=True)
DB = WORK / 'sessions.db'
DB.unlink(missing_ok=True)

## 1 · Why sessions

In notebook 02 you carried history by hand: `result.to_input_list() + [new
message]`. That works, but in a real app the history has to be stored somewhere
between requests, per user, and loaded again. A **session** does exactly that:

```
 Runner.run(agent, "What do I build?", session=SQLiteSession("user-ada", "sessions.db"))
    1. load the stored history for "user-ada"          ◄── database
    2. run the agent on  history + new message
    3. after each turn, append the new items            ──► database
```

You send only the new message; the session supplies the rest.

## 2 · Conversation memory with `SQLiteSession`

In [ ]:
chat = Agent(name='Assistant', model=model, instructions='You are brief and friendly.')

ada = SQLiteSession('user-ada', DB)          # session id + database file
bo = SQLiteSession('user-bo', DB)

print((await Runner.run(chat, 'Hi, I am Ada and I build delivery robots.', session=ada)).final_output)
print((await Runner.run(chat, 'What do I build?', session=ada)).final_output)
print((await Runner.run(chat, 'What do I build?', session=bo)).final_output, '  ← different session')

Now simulate a restart: new session objects, same database file:

In [ ]:
del ada
ada_again = SQLiteSession('user-ada', DB)
print((await Runner.run(chat, 'Remind me: what is my name?', session=ada_again)).final_output)
print('items stored for Ada:', len(await ada_again.get_items()))

## 3 · Working with the stored history

A session is a small API over a list of items:

| Method | Use |
|---|---|
| `await s.get_items(limit=None)` | read the history (e.g. to show a transcript) |
| `await s.add_items([...])` | append items yourself (e.g. import an old conversation) |
| `await s.pop_item()` | remove and return the most recent item: **undo** |
| `await s.clear_session()` | forget everything for this session id |

**Undo / correct a message.** The user made a typo and wants to rephrase. Pop the
assistant's reply and the user's message, then run again:

In [ ]:
s = SQLiteSession('undo-demo', DB)
await Runner.run(chat, 'Convert 100 EUR to USD at 1.08.', session=s)
print('before:', [ (i.get('role'), str(i.get('content'))[:40]) for i in await s.get_items()])

await s.pop_item()                       # the assistant's answer
await s.pop_item()                       # the user's message
result = await Runner.run(chat, 'Convert 250 EUR to USD at 1.08.', session=s)
print('after :', [ (i.get('role'), str(i.get('content'))[:40]) for i in await s.get_items()])

**Limiting how much history is sent.** Long conversations grow the prompt (and
the bill) every turn. `SessionSettings(limit=N)` stores everything but sends only
the most recent `N` items to the model:

In [ ]:
short_memory = SQLiteSession('limited', DB, session_settings=SessionSettings(limit=4))
for fact in ['My favourite colour is teal.', 'I live in Utrecht.', 'I have a dog called Bolt.']:
    await Runner.run(chat, fact, session=short_memory)
answer = await Runner.run(chat, 'What is my favourite colour?', session=short_memory)
stored = len(await SQLiteSession('limited', DB).get_items())      # a session object without the limit
print(answer.final_output, f'\n(stored: {stored} items, sent to the model: the last 4)')

The colour was mentioned six items ago, so it fell outside the window. (The
limit also applies to `get_items()` on that session object, which is why we
counted through a second object without the setting.) That's
the trade-off: a tight limit is cheap and forgetful. For long-lived assistants,
summarising old turns is the usual middle ground.

## 4 · Sessions and handoffs

A session stores the **conversation**, not *which agent* was active. After a
handoff (notebook 06) the next message should go to the new owner, so store
`result.last_agent.name` alongside the session id and start the next run from
that agent. The session supplies the shared history either way.

## 5 · What survives a crash?

Now the important question. A run makes several model calls and tool calls. If
the process is killed halfway through, what's in the session afterwards?

The notebook writes a small script whose agent must call `fetch_notes`, then
`write_report`. With `CRASH=1`, the process is killed instantly (`os._exit`,
like `kill -9`) **inside** `write_report`. Every tool logs its name to
`jobs.log`, so we can see what re-ran.

In [ ]:
(WORK / 'job.py').write_text(textwrap.dedent(f"""
    import asyncio, os, sys
    sys.path.insert(0, {str(SECTION)!r})
    from agents import Agent, Runner, SQLiteSession, function_tool
    from common import get_model

    @function_tool(description_override='Fetch research notes on a topic.')
    def fetch_notes(topic: str) -> str:
        open('jobs.log', 'a').write('fetch_notes ')
        return f'notes on {{topic}}: supplier output down 40%'

    @function_tool(description_override='Write the final report.')
    def write_report(text: str) -> str:
        open('jobs.log', 'a').write('write_report ')
        if os.environ.get('CRASH'):
            os._exit(1)                                  # killed mid-tool
        return 'report saved'

    agent = Agent(name='Researcher', model=get_model(), tools=[fetch_notes, write_report],
                  instructions='First call fetch_notes for "batteries", then call write_report with a '
                               'one-line summary, then answer briefly.')

    async def main(mode):
        session = SQLiteSession('job-1', 'job_session.db')
        if mode != 'show':
            prompt = 'Do the research job.' if mode == 'run' else 'Continue the job where it stopped.'
            result = await Runner.run(agent, prompt, session=session)
            print('finished:', result.final_output[:70])
        kinds = [item.get('type') or item.get('role') for item in await session.get_items()]
        print('session now holds:', kinds)

    asyncio.run(main(sys.argv[1]))
"""))

def job(mode, **env):
    import os
    out = subprocess.run([sys.executable, 'job.py', mode], cwd=WORK, text=True,
                         capture_output=True, env={**os.environ, **env})
    print(f'$ python job.py {mode}   exit={out.returncode}')
    if out.stdout.strip():
        print('  ' + out.stdout.strip().replace('\n', '\n  '))

for f in ('job_session.db', 'jobs.log'):
    (WORK / f).unlink(missing_ok=True)

job('run', CRASH='1')          # process 1: killed inside write_report
job('show')                    # a new process looks at the session

The session holds the user message **and the completed `fetch_notes` call with
its result**, but nothing of the `write_report` call that was running when the
process died. Sessions are written **after each turn**, not only at the end of
the run. Completed turns survive; the in-flight one is lost.

So can we continue? Run again with the same session and ask the agent to
continue:

In [ ]:
job('resume')
print('\ntools executed across both processes:', (WORK / 'jobs.log').read_text())

`fetch_notes` did **not** run again: the model saw its result in the history and
went straight on to `write_report`, which ran a second time because it was in
flight when the process died.

What this means in practice:

- **Turn-level durability comes for free with a session.** Completed tool calls
  and model turns are kept.
- **The in-flight step re-runs**, so tools with side effects must be
  **idempotent** (check "already saved?" before saving).
- **The resume is model-driven.** It worked because the model read the history
  and chose not to refetch. That's likely, not guaranteed. A graph framework
  resumes from an exact checkpoint instead of asking the model what to do next.

## 6 · The three kinds of state in the SDK

| State | What it holds | Saved when | Use it for |
|---|---|---|---|
| **Session** | the conversation items | after every turn, automatically | chat memory; recovering after a crash (model-driven) |
| **`RunState`** | a paused run, including pending approvals | when *you* call `to_json()` | human approvals (notebook 08) |
| **Context** | your own objects (user, db handle, counters) | never; lives in memory | per-run dependencies (notebook 02) |

What the SDK does **not** give you is an automatic, exact checkpoint of a
multi-step *code* workflow (notebook 05). If your Python orchestrates five agent
calls and crashes after the third, a rerun starts at the first, unless you save
each step's result and skip completed steps. That pattern is simple:

```python
async def step(name, make_result):
    path = CHECKPOINTS / f'{name}.json'
    if path.exists():
        return json.loads(path.read_text())          # done before the crash: reuse
    result = await make_result()
    path.write_text(json.dumps(result))              # durable before moving on
    return result

plan = await step('plan', lambda: plan_research(question))
```

The project uses exactly this. If you need it everywhere, with retries,
timers and signals, that's what durable-execution engines like Temporal are for.

**Other session backends.** `SQLiteSession` is built in. The SDK also ships
optional backends in `agents.extensions.memory` (SQLAlchemy, Redis, MongoDB,
Dapr, an encrypted wrapper, …) and `OpenAIConversationsSession`, which stores
history on OpenAI's servers. All implement the same small interface, so
switching is one line.

## 7 · Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Passing history manually **and** a session | duplicated messages | with a session, send only the new message |
| One session id for several users | users see each other's history | one session per conversation |
| Unbounded history | slow, expensive turns | `SessionSettings(limit=…)` or summarising |
| Assuming the whole run is saved at the end | surprises after crashes | sessions save per turn; plan for the in-flight step |
| Non-idempotent side-effecting tools | duplicates after a crash | check before acting |
| Expecting sessions to remember the active agent | replies come from triage again | store `last_agent` yourself |

## 8 · Check yourself

<details><summary><b>1.</b> With a session, what should each <code>Runner.run</code> call send as input?</summary>

Only the new user message. The session loads the stored history before the run and saves the new items afterwards.
</details>

<details><summary><b>2.</b> How do you let a user correct their last message?</summary>

`pop_item()` twice (the assistant's reply, then the user's message), then run again with the corrected message.
</details>

<details><summary><b>3.</b> The process was killed during the second tool call of a run. What's in the session?</summary>

Everything from completed turns: the user message, and the first tool call with its result. The in-flight turn (the second tool call) is lost, because sessions save after each turn.
</details>

<details><summary><b>4.</b> Why did <code>write_report</code> run twice, and what does that require of the tool?</summary>

It was executing when the process died, so its turn was never saved, and the resumed run called it again. Tools with side effects must be idempotent.
</details>

<details><summary><b>5.</b> Why is resuming from a session "likely, not guaranteed" to skip completed work?</summary>

The model decides what to do next after reading the history. It usually sees completed results and continues, but nothing forces it. A graph checkpoint records exactly which step comes next.
</details>

<details><summary><b>6.</b> Which kind of SDK state would you use for: chat memory, a pending refund approval, the current user's permissions?</summary>

A session; a `RunState` saved with `to_json()`; the context object.
</details>

## Takeaway

A **session** gives agents conversation memory, saved after every turn and keyed
by an id. Use it with only the new message as input, trim what's sent, and store
the active agent yourself. After a crash, completed turns are kept and the
in-flight step re-runs, so make side effects idempotent. For exact, step-level
recovery of code workflows, save each step's result yourself (or use a durable
engine).

Next → `10_streaming_tracing_and_hooks.ipynb`: watching runs as they happen.